# Material demand from electricity sector

In [1]:
from shared.utils import run_pathway
import pandas as pd
import plotly.express as px
import os

In [2]:
results = run_pathway('run_elec')

[run_pathway] Window 1/1 done in 137.2s
[run_pathway] Total time: 137.2s


In [3]:
periods = ['2020_2025', '2025_2030', '2030_2035', '2035_2040', '2040_2045', '2045_2050']
years = ['2025', '2030', '2035', '2040', '2045', '2050']
technologies = [t for t in results['F_new'].loc['2020_2025'].index
                if any(results['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]

elec_keywords = ['PV_', 'WIND_', 'HYDRO', 'NUCLEAR', 'CCGT', 'COAL_', 'OCGT_', 'TIDAL', 'GEOTHERMAL', 'AFC', 'PAFC', 'PEMFC', 'SOFC', 'WAVE']

elec_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in elec_keywords) and not t.startswith(('COAL_GAS', 'HYDRO_STORAGE', 'UNMINEABLE_COAL_SEAM'))]

elec_techs_positive = [t for t in elec_techs
                       if any(results['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]


In [4]:
elec_techs_positive

['NEW_HYDRO_DAM', 'PV_ROOF', 'WIND_ONSHORE']

In [5]:
df_plot = pd.DataFrame(
    {period: results['F_new'].loc[period].loc[elec_techs_positive].squeeze() for period in periods},
    index=elec_techs_positive
)

df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Technologies', value_name='Capacité'
)

fig = px.bar(df_melted, x='Période', y='Capacité', color='Technologies', barmode='stack')
fig.update_layout(xaxis_title='Période', yaxis_title='Capacité [GW]')
fig.show()

# Charger intensité matérielle

In [6]:
# Charger les intensités
df_mi = pd.read_excel('/Users/Paolo/Documents/PdM_code/Material_intensities.xlsx', sheet_name='MI_Energy', index_col=0)
df_ms = pd.read_excel('/Users/Paolo/Documents/PdM_code/Material_intensities.xlsx', sheet_name='MS_Energy_Disag')


In [7]:
tech_mapping = {
    'WIND_ONSHORE': 'Wind_GB-DFIG_SCIG_Onshore',
    'PV_ROOF':      'Sol_C-si_Silver',
    'NEW_HYDRO_DAM':'Hydro'
    }
tech_groups = {
    'WIND_ONSHORE': {
        'iam_source': 'Capacity..Electricity..Wind..Onshore',
        'excel_techs': ['Wind_DD-EESG_Onshore', 'Wind_GB-DFIG_SCIG_Onshore', 'Wind_DD-PMSG_Onshore', 'Wind_GB-PMSG_Onshore']
    },
    'PV_ROOF': {
        'iam_source': 'Capacity..Electricity..Solar..PV',
        'excel_techs': ['Sol_C-si_Silver', 'Sol_C-si_Copper', 'Sol_CdTe', 'Sol_CIGS', 'Sol_a-SiGe']
    },
    'NEW_HYDRO_DAM': {
        'iam_source': None,
        'excel_techs': ['Hydro']
    }
}
period_to_decades = {
    '2020_2025': (2020, None),
    '2025_2030': (2020, 2030),
    '2030_2035': (2030, None),
    '2035_2040': (2030, 2040),
    '2040_2045': (2040, None),
    '2045_2050': (2040, 2050),
}

In [8]:
def get_ms(iam_source, period):
    d1, d2 = period_to_decades[period]
    if iam_source is None:
        return pd.Series({'Hydro': 1.0})
    ms = df_ms[df_ms['IAM_Energy_Sources'] == iam_source].set_index('Decade').drop(columns=['IAM_Energy_Sources'])
    ms = ms.apply(pd.to_numeric, errors='coerce')
    if d2 is None:
        return ms.loc[d1]
    return (ms.loc[d1] + ms.loc[d2]) / 2

In [9]:
def plot_material_demand(material, cumulative=False, market_share=False, show = False):

    data = {}
    for period in periods:
        data[period] = {}
        if market_share:
            for model_tech, info in tech_groups.items():
                f_new = results['F_new'].loc[period].loc[model_tech].squeeze()
                ms = get_ms(info['iam_source'], period)
                for excel_tech in info['excel_techs']:
                    data[period][excel_tech] = f_new * ms[excel_tech] * df_mi.loc[material, excel_tech]
        else:
            for tech, excel_col in tech_mapping.items():
                data[period][tech] = results['F_new'].loc[period].loc[tech].squeeze() * df_mi.loc[material, excel_col]

    df_plot = pd.DataFrame(data)

    if cumulative:
        df_cumulative = df_plot.sum().cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title=f'Demande cumulative en {material}',
                      labels={'x': 'Période', 'y': 'Demande cumulative [t]'})
    else:
        df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Technologies', value_name='Demande [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Technologies', barmode='stack',
                     title=f'Demande en {material}')
        fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')
    if show:
        fig.show()

    #return df_plot

# Exemple
plot_material_demand('Copper', cumulative=False, market_share=True, show = True)

In [10]:
def plot_all_materials_demand(cumulative=False, market_share=False, show = False):
    
    data_all = {}
    for material in df_mi.index:
        total_by_period = []
        for period in periods:
            total = 0
            if market_share:
                for model_tech, info in tech_groups.items():
                    f_new = results['F_new'].loc[period].loc[model_tech].squeeze()
                    ms = get_ms(info['iam_source'], period)
                    for excel_tech in info['excel_techs']:
                        total += f_new * ms[excel_tech] * df_mi.loc[material, excel_tech]
            else:
                for tech, excel_col in tech_mapping.items():
                    total += results['F_new'].loc[period].loc[tech].squeeze() * df_mi.loc[material, excel_col]
            total_by_period.append(total)
        data_all[material] = total_by_period

    df_all = pd.DataFrame(data_all, index=periods)
    df_all = df_all.loc[:, df_all.max() > 0]

    if cumulative:
        df_cumulative = df_all.sum(axis=1).cumsum()
        fig = px.line(x=periods, y=df_cumulative.values,
                      title='Demande cumulative - Tous matériaux',
                      labels={'x': 'Période', 'y': 'Demande cumulative [t]'})
    else:
        df_all_annual = df_all / 5
        df_all_annual.index = years 
        df_melted = df_all_annual.reset_index().rename(columns={'index': 'Période'}).melt(
            id_vars='Période', var_name='Matériau', value_name='Demande [t]'
        )
        fig = px.bar(df_melted, x='Période', y='Demande [t]', color='Matériau', barmode='stack',
                     title='Demande en matériaux - Énergie')
        fig.update_layout(xaxis_title='Période', yaxis_title='Demande [t]')
    if show:
        fig.show()
    return df_all

#plot_all_materials_demand(cumulative=True,market_share=True)
plot_all_materials_demand(market_share=True, show = True)

,Aluminum,Boron,Cadmium,Chromium,Cobalt,Concrete,Copper,Dysprosium,Gallium,Germanium,...,Nickel,Polymers,Praesodymium,Selenium,Silicon,Silver,Tellurium,Terbium,Tin,Zinc
2020_2025,1042.539366,1.069851,0.004928,426.429560,0.341121,2.868263e+05,2345.221324,4.731474,0.000404,0.000340,...,325.864405,3862.216,6.468160,0.002735,9.006822,0.067543,0.005211,1.277989,0.78352,4596.904000
2025_2030,174.137878,0.000000,0.000000,76.825534,0.000000,0.000000e+00,161.845792,0.000000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,20.486809
2030_2035,25979.937168,10.693211,0.790005,10829.546605,3.141217,2.613581e+06,34592.229352,44.579131,0.072937,0.074988,...,2961.967351,37416.552,61.618427,0.494351,1096.926816,8.225194,0.835408,12.439288,97.24944,43714.801191
2035_2040,992.556445,1.190063,0.007084,406.188052,0.321627,2.694459e+05,2251.653880,4.831658,0.000680,0.000736,...,306.376598,3650.616,6.810921,0.004607,8.753181,0.065632,0.007491,1.379221,0.78352,4343.904000
2040_2045,987.044572,1.267731,0.007802,407.703817,0.316479,2.687118e+05,2308.016605,5.025826,0.000772,0.000868,...,304.673032,3650.616,7.211785,0.005231,8.668634,0.064994,0.008251,1.464404,0.78352,4343.904000
2045_2050,17219.501247,1.431679,0.008521,7571.438852,0.328892,2.837274e+05,17535.299423,5.542076,0.000864,0.001001,...,320.728514,3862.216,8.076841,0.005854,8.584087,0.064357,0.009010,1.646411,0.78352,6500.506687


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_all = plot_all_materials_demand(market_share=True)

def plot_all_materials_demand_sub(save=False):
    materials_positive = df_all.columns.tolist()
    n = len(materials_positive)
    ncols = 6
    nrows = -(-n // ncols)  # arrondi supérieur

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=materials_positive)

    for i, material in enumerate(materials_positive):
        row = i // ncols + 1
        col = i % ncols + 1
        
        df_all_annual = df_all / 5
        df_all_annual.index = years
        
        fig.add_trace(
            go.Bar(x=years, y=df_all_annual[material].values, name=material, showlegend=False),
            row=row, col=col
        )

    fig.update_layout(height=300 * nrows, title='Demande annuelle par matériau - Énergie')
    fig.update_yaxes(title_text='[t]', col=1)    
    fig.show()

    if save:
        save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy')
        os.makedirs(save_dir, exist_ok=True)

        #filepath = os.path.join(save_dir, 'tot_mat_elec.png')
        #fig.write_image(filepath, width=250*ncols, height=300*nrows)

plot_all_materials_demand_sub(save=True)

In [ ]:
def save_material_plots_elec(save_dir='~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy'):
    save_dir = os.path.expanduser(save_dir)
    os.makedirs(save_dir, exist_ok=True)

    for material in df_mi.index:
        data = {}
        for period in periods:
            data[period] = {}
            for model_tech, info in tech_groups.items():
                f_new = results['F_new'].loc[period].loc[model_tech].squeeze()
                ms = get_ms(info['iam_source'], period)
                for excel_tech in info['excel_techs']:
                    data[period][excel_tech] = f_new * ms[excel_tech] * df_mi.loc[material, excel_tech]

        df_mat = pd.DataFrame(data)
        df_mat = df_mat / 5  # demande annuelle

        if df_mat.max().max() <= 0:
            continue

        df_mat.columns = years

        df_melted = df_mat.T.reset_index().rename(columns={'index': 'Année'}).melt(
            id_vars='Année', var_name='Technologie', value_name='Demande [t/an]'
        )

        fig = px.bar(df_melted, x='Année', y='Demande [t/an]', color='Technologie', barmode='stack',
                     title=f'Demande en {material} - Électricité')
        fig.update_layout(xaxis_title='Année', yaxis_title='Demande [t/an]')

        filepath = os.path.join(save_dir, f'{material}.png')
        #fig.write_image(filepath)
        #print(f'Saved: {filepath}')

save_material_plots_elec()

Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Aluminum.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Boron.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Cadmium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Chromium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Cobalt.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Concrete.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Copper.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Dysprosium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Gallium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDo

Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Nickel.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Polymers.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Praesodymium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Selenium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Silicon.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Silver.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Tellurium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Terbium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy/Tin.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudD

# --------------------------------------- Mobility demand ---------------------------------------------

In [13]:
priv_mob_keywords = ['CAR_', 'SUV_']

priv_mob_techs = [t for t in results['F_new'].loc['2020_2025'].index 
              if any(kw in t for kw in priv_mob_keywords) and not t.endswith(('_LD', '_MD', '_SD')) ]

print(priv_mob_techs)

priv_mob_techs_positive = [t for t in priv_mob_techs
                       if any(results['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]

print(priv_mob_techs_positive)



['CAR_BIODIESEL_B100', 'CAR_BIODIESEL_B100_ELD', 'CAR_BIODIESEL_B20', 'CAR_BIODIESEL_B20_ELD', 'CAR_BIOETOH_E10', 'CAR_BIOETOH_E10_ELD', 'CAR_BIOETOH_E85', 'CAR_BIOETOH_E85_ELD', 'CAR_BIOMEOH', 'CAR_BIOMEOH_ELD', 'CAR_CNG', 'CAR_CNG_ELD', 'CAR_CSNG', 'CAR_CSNG_ELD', 'CAR_DIESEL', 'CAR_DIESEL_ELD', 'CAR_ETOH_E10', 'CAR_ETOH_E10_ELD', 'CAR_ETOH_E85', 'CAR_ETOH_E85_ELD', 'CAR_EV', 'CAR_EV_ELD', 'CAR_FC_H2', 'CAR_FC_H2_ELD', 'CAR_GASOLINE', 'CAR_GASOLINE_ELD', 'CAR_HEV', 'CAR_HEV_ELD', 'CAR_MEOH', 'CAR_MEOH_ELD', 'CAR_PHEV', 'CAR_PHEV_ELD', 'CAR_PROPANE', 'CAR_PROPANE_ELD', 'SUV_BIODIESEL_B100', 'SUV_BIODIESEL_B100_ELD', 'SUV_BIODIESEL_B20', 'SUV_BIODIESEL_B20_ELD', 'SUV_BIOETOH_E10', 'SUV_BIOETOH_E10_ELD', 'SUV_BIOETOH_E85', 'SUV_BIOETOH_E85_ELD', 'SUV_CNG', 'SUV_CNG_ELD', 'SUV_CSNG', 'SUV_CSNG_ELD', 'SUV_DIESEL', 'SUV_DIESEL_ELD', 'SUV_ETOH_E10', 'SUV_ETOH_E10_ELD', 'SUV_ETOH_E85', 'SUV_ETOH_E85_ELD', 'SUV_EV', 'SUV_EV_ELD', 'SUV_FC_H2', 'SUV_FC_H2_ELD', 'SUV_GASOLINE', 'SUV_GASOLINE_ELD

In [14]:
df_plot = pd.DataFrame(
    {period: results['F_new'].loc[period].loc[priv_mob_techs_positive].squeeze() for period in periods},
    index=priv_mob_techs_positive
)

df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
    id_vars='Période', var_name='Technologies', value_name='Capacité'
)

fig = px.bar(df_melted, x='Période', y='Capacité', color='Technologies', barmode='stack')
fig.update_layout(xaxis_title='Période', yaxis_title='Mpkm/h')
fig.show()

In [15]:
ref_size_priv_mob = 54.392


data_vehicles = {}
for period in periods:
    data_vehicles[period] = {
        tech: results['F_new'].loc[period].loc[tech].squeeze()  * 1e6 / ref_size_priv_mob
        for tech in priv_mob_techs_positive
    }

df_priv_mob = pd.DataFrame(data_vehicles)

In [16]:
# passer en format long
df_long = df_priv_mob.reset_index().melt(
    id_vars='index',
    var_name='Period',
    value_name='Vehicles'
).rename(columns={'index': 'Technology'})

fig = px.bar(
    df_long,
    x='Period',
    y='Vehicles',
    color='Technology',
    title='Number of new private vehicles per period',
)

fig.show()

In [17]:
xl = '/Users/Paolo/Documents/PdM_code/Material_intensities.xlsx'

# --- Chargement MI_EV ---
df_raw = pd.read_excel(xl, sheet_name='MI_EV', header=None)

# Body
df_body = pd.read_excel(xl, sheet_name='MI_EV', header=1, nrows=5, usecols='A:D').set_index('Metal')

# Battery (g/kWh)
df_batt_mi = pd.read_excel(xl, sheet_name='MI_EV', header=10, nrows=5)
df_batt_mi = df_batt_mi.rename(columns={'Metal intensity by battery chemistry': 'Metal'}).set_index('Metal')
batt_cols = ['Gr-LMO', 'Gr-NMC', 'Gr-LMFP', 'Gr-NCA', 'Gr-LFP', 'Na-ion']
df_batt_mi = df_batt_mi[batt_cols].apply(pd.to_numeric, errors='coerce')

# Motor (g/vehicle)
df_motor_mi = pd.read_excel(xl, sheet_name='MI_EV', header=19, nrows=5, usecols='A:C')
df_motor_mi = df_motor_mi.rename(columns={'Metal': 'Metal'}).set_index(df_motor_mi.columns[0])

# Battery size (kWh)
df_batt_size = pd.read_excel(xl, sheet_name='MI_EV', header=28, nrows=1, usecols='A:D').set_index('Vehicle part')

# --- Chargement MS Battery ---
df_ms_raw = pd.read_excel(xl, sheet_name='MS_Battery_Motor_LDV', header=1, nrows=7)
df_ms_batt = df_ms_raw.rename(columns={
    'MS_Battery : market share by battery type': 'Battery_type',
    'Unnamed: 1': 'Scenario'
}).set_index('Battery_type').drop(columns=['Scenario', 'Unnamed: 21', 'Unnamed: 22'], errors='ignore')
df_ms_batt = df_ms_batt.apply(pd.to_numeric, errors='coerce')
df_ms_batt.columns = [int(float(c)) if str(c).replace('.0','').isdigit() else c for c in df_ms_batt.columns]
df_ms_batt = df_ms_batt[[c for c in df_ms_batt.columns if isinstance(c, int)]]

# Motor MS
ms_PM = 0.876289
ms_Ind = 0.123711

# Mapping tech -> vehicle type
tech_to_vtype = {
    'CAR_EV': 'BEV', 'SUV_EV': 'BEV',
    'CAR_PHEV': 'PHEV', 'SUV_PHEV_GASOLINE': 'PHEV',
    'CAR_HEV': 'HyEV', 'SUV_HY_GASOLINE': 'HyEV',
    'CAR_DIESEL': 'ICEV', 'CAR_GASOLINE': 'ICEV',
    'CAR_CNG': 'ICEV', 'SUV_DIESEL': 'ICEV', 'SUV_GASOLINE': 'ICEV'
}

vtype_to_body_col = {'ICEV': 'ICEV', 'PHEV': 'PHEV', 'BEV': 'BEV', 'HyEV': 'ICEV'}
vtype_to_batt_size = {'BEV': 62.5, 'PHEV': 21.8, 'HyEV': 1.3, 'ICEV': 0}
vtype_to_motor_power = {'BEV': 72, 'PHEV': 68, 'HyEV': 50, 'ICEV': 0}
motor_ref_power = 72  

period_to_years = {
    '2020_2025': (2025, None),
    '2025_2030': (2030, None),
    '2030_2035': (2030, 2040),
    '2035_2040': (2040, None),
    '2040_2045': (2040, 2050),
    '2045_2050': (2040, None),
}

In [18]:
def get_batt_ms(period):
    y1, y2 = period_to_years[period]
    if y2 is None:
        return df_ms_batt[y1]
    return (df_ms_batt[y1] + df_ms_batt[y2]) / 2

def get_vehicle_mi(material, vtype, period):
    # Body
    body_col = vtype_to_body_col[vtype]
    mi = df_body.loc[material, body_col] if material in df_body.index else 0

    # Battery
    batt_size = vtype_to_batt_size[vtype]
    if batt_size > 0 and material in df_batt_mi.index:
        batt_ms = get_batt_ms(period)
        mi += sum(df_batt_mi.loc[material, bc] * batt_ms.loc[bc] for bc in batt_cols) * batt_size

    # Motor (proportionnel à la puissance du moteur)
    motor_power = vtype_to_motor_power[vtype]
    if vtype != 'ICEV' and material in df_motor_mi.index:
        motor_mi = df_motor_mi.loc[material, 'PM'] * ms_PM + df_motor_mi.loc[material, 'Ind'] * ms_Ind
        mi += motor_mi * (motor_power / motor_ref_power)

    return mi  # g/vehicle

In [19]:
def plot_all_materials_demand_mob(cumulative=False, show=False):
    
    data_all = {}
    for material in df_body.index.tolist() + df_batt_mi.index.tolist() + df_motor_mi.index.tolist():
        total_by_period = []
        for period in periods:
            total = 0
            for tech in priv_mob_techs_positive:
                vtype = tech_to_vtype[tech]
                n_veh = results['F_new'].loc[period].loc[tech].squeeze() * 1e6 / ref_size_priv_mob
                mi = get_vehicle_mi(material, vtype, period)  # g/vehicle
                total += n_veh * mi / 1e6  # -> tonnes
            total_by_period.append(total / 5)
        data_all[material] = total_by_period

    df_all_mob = pd.DataFrame(data_all, index=years)
    df_all_mob = df_all_mob.loc[:, df_all_mob.max() > 0]

    df_melted = df_all_mob.reset_index().rename(columns={'index': 'Année'}).melt(
        id_vars='Année', var_name='Matériau', value_name='Demande [t/an]'
    )
    fig = px.bar(df_melted, x='Année', y='Demande [t/an]', color='Matériau', barmode='stack',
                 title='Demande annuelle en matériaux - Mobilité privée')
    fig.update_layout(xaxis_title='Année', yaxis_title='Demande [t/an]')
    if show:
        fig.show()
    return df_all_mob

df_all_mob = plot_all_materials_demand_mob(cumulative=False, show=True)

In [ ]:
materials_positive = df_all_mob.columns.tolist()
n = len(materials_positive)
ncols = 6
nrows = -(-n // ncols)

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=materials_positive)

for i, material in enumerate(materials_positive):
    row = i // ncols + 1
    col = i % ncols + 1
    fig.add_trace(
        go.Bar(x=years, y=(df_all_mob[material]).values, name=material, showlegend=False),
        row=row, col=col
    )

fig.update_yaxes(title_text='[t/an]', col=1)
fig.update_layout(height=300 * nrows, title='Demande annuelle par matériau - Mobilité privée')
fig.show()

save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob')
os.makedirs(save_dir, exist_ok=True)

#filepath = os.path.join(save_dir, 'tot_mat_priv_mob.png')
#fig.write_image(filepath, width=250*ncols, height=300*nrows)

# Save plots

In [ ]:
materials_list = df_body.index.tolist() + df_batt_mi.index.tolist() + df_motor_mi.index.tolist()
materials_list = list(dict.fromkeys(materials_list))  # enlever doublons en gardant l'ordre

for material in materials_list:
    data = {}
    for period in periods:
        data[period] = {}
        for tech in priv_mob_techs_positive:
            vtype = tech_to_vtype[tech]
            n_veh = df_priv_mob.loc[tech, period]
            mi = get_vehicle_mi(material, vtype, period)
            data[period][tech] = n_veh * mi / (5*1e6)  # tonnes

    df_mat = pd.DataFrame(data)
    df_mat.columns = years

    if df_mat.max().max() <= 0:
        continue  # skip matériaux sans demande

    df_melted = df_mat.T.reset_index().rename(columns={'index': 'Année'}).melt(
        id_vars='Année', var_name='Technologie', value_name='Demande [t/an]'
    )

    fig = px.bar(df_melted, x='Année', y='Demande [t/an]', color='Technologie', barmode='stack',
                 title=f'Demande en {material} - Mobilité privée')
    fig.update_layout(xaxis_title='Année', yaxis_title='Demande [t/an]')

    #filepath = os.path.join(save_dir, f'{material}.png')
    #fig.write_image(filepath)
    #print(f'Saved: {filepath}')

Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Aluminum.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Chromium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Lead.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Iron.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Zinc.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Cobalt.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Copper.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Lithium.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Manganese.png
Saved: /Users/Paolo/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Priv_mob/Nickel.pn

# ------------------------ Mobility live fleet ----------------------

In [22]:
years_live = ['YEAR_2020', 'YEAR_2025', 'YEAR_2030', 'YEAR_2035', 'YEAR_2040', 'YEAR_2045', 'YEAR_2050']

priv_mob_techs_live = [t for t in results['F_Mult'].loc['YEAR_2020'].index 
              if any(kw in t for kw in priv_mob_keywords) and not t.endswith(('_LD', '_MD', '_SD')) ]

print(priv_mob_techs_live)

priv_mob_techs_live_positive = [t for t in priv_mob_techs_live
                       if any(results['F_Mult'].loc[year].loc[t].squeeze() > 0 for year in years_live)]

print(priv_mob_techs_live_positive)

['CAR_DIESEL', 'CAR_EV', 'CAR_GASOLINE', 'CAR_HEV', 'CAR_PHEV', 'SUV_DIESEL', 'SUV_EV', 'SUV_GASOLINE', 'SUV_HY_GASOLINE', 'SUV_PHEV_GASOLINE']
['CAR_DIESEL', 'CAR_EV', 'CAR_GASOLINE', 'CAR_HEV', 'CAR_PHEV', 'SUV_DIESEL', 'SUV_EV', 'SUV_GASOLINE', 'SUV_HY_GASOLINE', 'SUV_PHEV_GASOLINE']


In [23]:
data_vehicles_live = {}
for year in years_live:
    techs_available = results['F_Mult'].loc[year].index
    data_vehicles_live[year] = {
        tech: (results['F_Mult'].loc[year].loc[tech].squeeze() * 1e6 / ref_size_priv_mob if tech in techs_available else 0)
        for tech in priv_mob_techs_live_positive
    }

df_priv_mob_live = pd.DataFrame(data_vehicles_live)
print(df_priv_mob_live)

                      YEAR_2020     YEAR_2025     YEAR_2030     YEAR_2035  \
CAR_DIESEL         2.437588e+04  2.267712e+04  1.455183e+04  6.426538e+03   
CAR_EV             4.553230e+04  7.197522e+04  5.679778e+04  9.277013e+05   
CAR_GASOLINE       2.526813e+06  2.463919e+06  1.621648e+06  7.793770e+05   
CAR_HEV            3.955331e+04  4.634021e+04  3.315577e+04  1.997134e+04   
CAR_PHEV           3.449417e+04  3.943848e+04  2.794042e+04  1.644236e+04   
SUV_DIESEL         1.655720e+04  1.626837e+04  1.074930e+04  5.230238e+03   
SUV_EV             1.655720e+04  5.915771e+04  6.901676e+05  1.323858e+06   
SUV_GASOLINE       1.856706e+06  2.129678e+06  1.510776e+06  8.918737e+05   
SUV_HY_GASOLINE    2.575564e+04  4.634021e+04  3.775499e+04  2.916978e+04   
SUV_PHEV_GASOLINE  1.287782e+04  3.401569e+04  2.972308e+04  2.543047e+04   

                      YEAR_2040     YEAR_2045     YEAR_2050  
CAR_DIESEL         0.000000e+00  0.000000e+00  0.000000e+00  
CAR_EV             9.611737e

In [24]:
# passer en format long
df_long_live = df_priv_mob_live.reset_index().melt(
    id_vars='index',
    var_name='Year',
    value_name='Vehicles'
).rename(columns={'index': 'Technology'})

fig = px.bar(
    df_long_live,
    x='Year',
    y='Vehicles',
    color='Technology',
    title='Number of private vehicles per year',
)

fig.show()

# -------------------------------------------------------------------------------------

In [32]:
for period in periods:
    print(period, results['F_new'].loc[period].loc['CCGT_BIOGAS_CC'].squeeze())

2020_2025 0.0
2025_2030 0.0
2030_2035 0.0
2035_2040 0.0
2040_2045 0.0
2045_2050 0.0


In [28]:
pd.Series({period: results['F_new'].loc[period].loc['AFC'].squeeze() for period in periods})

2020_2025    0.0
2025_2030    0.0
2030_2035    0.0
2035_2040    0.0
2040_2045    0.0
2045_2050    0.0
dtype: float64

In [6]:
print(results['F_Mult'].loc['YEAR_2030'].loc['WIND_ONSHORE'].squeeze())

2.4136
